# B2-019 — Session 3: Multi-Head Attention, Position, and Cost

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).


## 1. Projection and head dimensions

With model width $d$ and $h$ heads, require $h\mid d$ and set $d_h=d/h$. Learned projections produce Q, K, V of width $d$, then reshape $(B,n,d)$ to $(B,h,n,d_h)$.

**Checkpoint 1A.** For $d=12,h=3$, state $d_h$.

**Checkpoint 1B.** Which permutation puts the head axis before sequence?

In [ ]:
import numpy as np
SEED = 20260808
def sinusoidal(length, width):
    positions = np.arange(length, dtype=np.float64)[:, None]
    frequencies = np.exp(-np.log(10000.0) * np.arange(0, width, 2) / width)
    table = np.zeros((length, width), dtype=np.float64)
    table[:, 0::2] = np.sin(positions * frequencies)
    table[:, 1::2] = np.cos(positions * frequencies[: table[:, 1::2].shape[1]])
    return table
assert np.array_equal(sinusoidal(3, 4)[0], [0., 1., 0., 1.])

## 2. Independent attention and concatenation

Each head builds a separate $(B,n,n)$ score matrix. After weighted values, transpose $(B,h,n,d_h)$ to $(B,n,h,d_h)$ and reshape to $(B,n,d)$.

**Worked example 1.** Two one-dimensional heads returning 3 and -2 concatenate to `(3,-2)` at the same token.

**Checkpoint 2A.** Why must concatenation use the feature direction rather than sequence?

**Checkpoint 2B.** What shape enters the output projection?

In [ ]:
B, n, d, h = 2, 5, 12, 3
dh = d // h
heads = np.zeros((B, h, n, dh))
concatenated = heads.transpose(0, 2, 1, 3).reshape(B, n, d)
assert concatenated.shape == (B, n, d)

## 3. Sinusoidal positional encoding

For zero-based position $p$ and even coordinate $2i$, use $\sin(p/10000^{2i/d})$; for odd coordinate $2i+1$, use $\cos(p/10000^{2i/d})$. Add the table to numeric token inputs before attention.

**Worked example 2.** At $p=0$, all sine coordinates are 0 and all cosine coordinates are 1.

**Checkpoint 3A.** Why does pure self-attention need an added order signal?

**Checkpoint 3B.** State the positional table shape for maximum length $L$.

## 4. Exact time and score-memory cost

Projection cost is $\Theta(Bnd^2)$. Standard attention score formation and weighted values cost $\Theta(Bn^2d)$ across all heads. Materialized scores or weights require $Bhn^2$ scalars; changing $h$ does not change the leading arithmetic when total width $d$ is fixed, but it scales score storage.

**Checkpoint 4A.** If $n$ doubles, by what factor does score memory change?

**Checkpoint 4B.** If $d$ doubles at fixed $n$, what happens to the attention-product term?

## 5. Common pitfalls and forward route

**Common pitfalls.** Broken: concatenate on the sequence axis, doubling length. Fix: restore `(B,n,h,d_h)` first. Broken: count only one head's score matrix. Fix: include $h$.

**Exam connections.** Round 2 cost questions distinguish arithmetic from materialized score memory and demand exact dimensions.

**Going deeper.** Session 4 turns the shape ledger into a tested module and tiny training loop.

Checkpoint answers: 1A 4; 1B transpose head and sequence axes; 2A heads describe features for the same token; 2B $(B,n,d)$; 3A attention alone is permutation equivariant; 3B $(L,d)$; 4A fourfold; 4B twofold.